# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 06

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. В чем практическая разница между обращением к данным через `.loc[]` и `.iloc[]` в библиотеке Pandas?
```python
val_a = df.loc[0, 'nickname']
val_b = df.iloc[0, 1]
```
По какому принципу находит элемент каждый из этих методов?

**Ответ:**
* `.loc[]` осуществляет доступ по **пользовательским именам и меткам** строк и колонок (label-based: название столбца 'nickname', исходная метка строки 0).
* `.iloc[]` осуществляет доступ строго по **целочисленным порядковым номерам позиций** (integer position-based: 0-я строка, 1-й по счету столбец).

---

#### 1.2. Аналитик выполнил две команды:
```python
cnt = df['city'].nunique()
lst = df['city'].unique()
```
Чем принципиально различаются результаты в переменных `cnt` и `lst`? Что возвращает число, а что — массив самих названий?

**Ответ:**
* `cnt = df['city'].nunique()` возвращает одно целое **число** — количество различных уникальных городов.
* `lst = df['city'].unique()` возвращает **массив/список** с самими уникальными названиями этих городов.

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["orders_share"] = (df["quantity"] / df["quantity"].sum() * 100).round(2)
```

**Ответ:**
Рассчитывает процентную долю количества проданных единиц конкретного товара в общем объеме всех продаж бара.

---

#### 1.4. Зачем после группировки данных в Pandas аналитики вызывают метод `.reset_index()`?
```python
summary = df.groupby('category')['total_amount'].sum().reset_index()
```
Как изменится структура таблицы `summary` при добавлении `.reset_index()`?

**Ответ:**
Метод `.reset_index()` переносит столбец группировки (`category`) из индекса обратно в обычные столбцы данных и возвращает таблице стандартный целочисленный индекс строк от 0. Без него `category` осталась бы индексом.

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `members.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_06")
df_audit = pd.read_csv(data_dir / "members.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Максимальный баланс участника: 2203.00")

Количество строк в таблице: 150
Столбцы датафрейма: ['member_id', 'nickname', 'email', 'city', 'registered_at', 'loyalty_tier', 'balance']
Максимальный баланс участника: 2203.00

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `game_sessions.csv` разделителем столбцов является точка с запятой (`;`). Без параметра `sep=';'` данные читаются в один столбец, что приводит к `KeyError`.
2. В словаре метода `.agg({...})` пропущена запятая между строками словаря, что вызывает ошибку компиляции `SyntaxError: invalid syntax`.

In [3]:
data_dir = pathlib.Path("game_sessions.csv").parent if os.path.exists("game_sessions.csv") else pathlib.Path(f"../variant_06")
df_data = pd.read_csv(data_dir / "game_sessions.csv", sep=";")

summary = df_data.groupby("game_title").agg({
    "total_paid": "sum",
    "duration_hours": "mean"
}).reset_index()

summary["duration_hours"] = summary["duration_hours"].round(2)
summary = summary.sort_values(by="total_paid", ascending=False).reset_index(drop=True)
display(summary)

,game_title,total_paid,duration_hours
0,Cyberpunk 2077,27807,2.05
1,Warzone,25713,2.20
2,Dota 2,25511,2.22
3,Atomic Heart,24037,1.99
4,Apex Legends,23220,2.18
5,Counter-Strike 2,22924,2.21
6,Rust,22046,2.16
7,Valorant,21458,2.10
8,PUBG,18238,2.21
9,GTA V,17081,2.22


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `game_sessions` колонка `member_id` является внешним ключом (`FOREIGN KEY`), ссылающимся на таблицу `members`.
Зачем нужна эта связь и что произойдет, если попытаться добавить игровую сессию для несуществующего клиента с `member_id = 99999`?

**Ответ:**
Внешний ключ связывает дочернюю таблицу (`game_sessions`) с родительской (`members`) и обеспечивает **ссылочную целостность**. СУБД отклонит вставку сессии с несуществующим клиентом (`FOREIGN KEY constraint failed`), предотвращая появление «потерянных» записей.

---

#### 3.2. Разработчик через скрипт на Python выполнил добавление новых компьютеров командой `cursor.execute("INSERT INTO hardware_pcs ...")`, но после закрытия программы обнаружил, что компьютеры не сохранились в файле базы данных.
Какую обязательную команду для объекта соединения (`conn`) он забыл вызвать и почему?

**Ответ:**
Он забыл вызвать команду фиксации транзакции `conn.commit()`. В библиотеке `sqlite3` изменения, внесенные через курсор, по умолчанию находятся во временной транзакции и не сохраняются на диске без явного `commit()`.

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT discount_promo, COUNT(*) AS cnt, AVG(duration_hours) AS avg_h, SUM(total_paid) AS rev FROM game_sessions GROUP BY discount_promo;
```

**Ответ:**
Запрос группирует сессии по наличию промокода (`discount_promo`) и рассчитывает число сессий, среднюю длительность и суммарную выручку клуба.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_06")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT session_id, member_id, game_title, duration_hours, total_paid, client_rating, session_date
FROM game_sessions
WHERE client_rating = 5 AND duration_hours >= 3.0
ORDER BY total_paid DESC, session_date DESC
LIMIT 5;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,session_id,member_id,game_title,duration_hours,total_paid,client_rating,session_date
0,20477,118,Warzone,4.0,1680.0,5.0,2026-02-05 10:45:00
1,20024,61,Dota 2,3.0,1260.0,5.0,2026-03-03 18:15:00
2,20314,7,Cyberpunk 2077,4.0,1080.0,5.0,2026-02-20 18:45:00
3,20367,109,PUBG,4.0,920.0,5.0,2026-01-31 09:30:00
4,20483,103,Warzone,5.0,850.0,5.0,2026-03-13 09:45:00


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT item_name, COUNT(*) AS orders_count, SUM(quantity) AS total_qty, ROUND(SUM(total_amount), 2) AS total_revenue
FROM bar_orders
GROUP BY item_name
HAVING SUM(quantity) >= 15
ORDER BY total_revenue DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,item_name,orders_count,total_qty,total_revenue
0,Мерч-футболка CyberClub,21,25,37500.0
1,Кепка CyberClub Team,24,35,28000.0
2,Коврик для мыши CyberClub Speed,20,30,27000.0
3,Сэндвич с ветчиной и сыром,28,45,11700.0
4,Бургер Классический,24,34,11560.0
5,Энергетик Adrenaline Rush,29,42,7980.0
6,Энергетик Red Bull,22,35,7700.0
7,Кофе Латте,29,42,7560.0
8,Чипсы картофельные Lays,24,39,6240.0
9,Минеральная вода Borjomi,24,38,5320.0


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT m.member_id, m.nickname, m.city, COUNT(b.order_id) AS drinks_count, ROUND(SUM(b.total_amount), 2) AS spent_on_drinks
FROM members m
JOIN bar_orders b ON m.member_id = b.member_id
WHERE b.category = 'Напитки'
GROUP BY m.member_id, m.nickname, m.city
ORDER BY spent_on_drinks DESC
LIMIT 10;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,member_id,nickname,city,drinks_count,spent_on_drinks
0,70,Flash_415,Москва,6,2360.0
1,131,Tempest_107,Москва,5,1700.0
2,74,Flash_962,Новосибирск,2,1120.0
3,126,Falcon_489,Москва,3,980.0
4,92,Phoenix_638,Москва,3,900.0
5,49,Turbo_605,Москва,1,880.0
6,104,Phantom_468,Санкт-Петербург,2,880.0
7,111,Rogue_127,Москва,3,830.0
8,82,Paladin_748,Санкт-Петербург,3,770.0
9,68,Kaiser_624,Москва,2,760.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT order_id, member_id, item_name, category, item_price, total_amount
FROM bar_orders
WHERE item_price > (SELECT AVG(item_price) FROM bar_orders)
ORDER BY item_price DESC
LIMIT 10;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,order_id,member_id,item_name,category,item_price,total_amount
0,6023,112,Мерч-футболка CyberClub,Мерч,1500.0,1500.0
1,6043,105,Мерч-футболка CyberClub,Мерч,1500.0,1500.0
2,6053,51,Мерч-футболка CyberClub,Мерч,1500.0,4500.0
3,6085,58,Мерч-футболка CyberClub,Мерч,1500.0,3000.0
4,6106,81,Мерч-футболка CyberClub,Мерч,1500.0,1500.0
5,6111,138,Мерч-футболка CyberClub,Мерч,1500.0,1500.0
6,6130,127,Мерч-футболка CyberClub,Мерч,1500.0,1500.0
7,6138,139,Мерч-футболка CyberClub,Мерч,1500.0,1500.0
8,6144,53,Мерч-футболка CyberClub,Мерч,1500.0,1500.0
9,6150,126,Мерч-футболка CyberClub,Мерч,1500.0,1500.0


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Сессии с учетом скидки

**Найденные ошибки:**
1. Использование псевдонима в `WHERE`: псевдоним `discounted_paid` еще не вычислен на этапе фильтрации строк, необходимо написать `total_paid * 0.9 > 400`.

In [9]:
query_51_fixed = '''
SELECT 
    session_id,
    member_id,
    game_title,
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE total_paid * 0.9 > 400
ORDER BY discounted_paid DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,session_id,member_id,game_title,discounted_paid
0,20155,97,Rust,1944.0
1,20125,94,Counter-Strike 2,1800.0
2,20393,52,Rust,1800.0
3,20174,62,Atomic Heart,1620.0
4,20477,118,Warzone,1512.0
5,20357,87,Cyberpunk 2077,1458.0
6,20130,86,GTA V,1242.0
7,20414,35,Dota 2,1242.0
8,20024,61,Dota 2,1134.0
9,20197,59,Cyberpunk 2077,1134.0


#### 5.2. Анализ крупных расходов клиентов в баре

**Найденные ошибки:**
1. Ошибка ключа соединения: `m.member_id = b.order_id` связывает идентификатор клиента с номером заказа вместо `b.member_id`.
2. Синтаксис и alias: предложение `WHERE` размещено после `GROUP BY` и использует псевдоним `total_bar`, необходимо написать `HAVING SUM(b.total_amount) > 1500`.

In [10]:
query_52_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    SUM(b.total_amount) AS total_bar
FROM members m
JOIN bar_orders b ON m.member_id = b.member_id
GROUP BY m.member_id, m.nickname, m.city
HAVING SUM(b.total_amount) > 1500
ORDER BY total_bar DESC
LIMIT 10;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,member_id,nickname,city,total_bar
0,51,Striker_625,Казань,7120.0
1,125,Zero_718,Москва,5190.0
2,56,Lynx_818,Москва,5040.0
3,122,Specter_522,Екатеринбург,4380.0
4,135,Kraken_952,Москва,4325.0
5,53,Paladin_538,Москва,4160.0
6,70,Flash_415,Москва,4100.0
7,102,Frost_990,Новосибирск,4005.0
8,58,Warlock_426,Санкт-Петербург,3820.0
9,50,Kaiser_112,Москва,3580.0


#### 5.3. Сравнение клиентов со средней длительностью сессий клуба

**Найденные ошибки:**
1. Порядок команд: блок `WHERE` написан после `GROUP BY`, фильтрация агрегатов должна выполняться через `HAVING`.
2. Использование псевдонима: в фильтрации указан alias `avg_member_hours` вместо агрегатной функции `AVG(s.duration_hours)`.
3. Многострочный подзапрос: наличие `GROUP BY game_title` в подзапросе возвращает вектор значений, несовместимый со скалярным оператором `>`.

In [11]:
query_53_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    ROUND(AVG(s.duration_hours), 2) AS avg_member_hours
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.member_id, m.nickname, m.city
HAVING AVG(s.duration_hours) > (
    SELECT AVG(duration_hours) 
    FROM game_sessions
)
ORDER BY avg_member_hours DESC
LIMIT 10;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,member_id,nickname,city,avg_member_hours
0,10,Mystic_193,Санкт-Петербург,5.00
1,11,Paladin_177,Екатеринбург,3.50
2,9,Phoenix_800,Москва,3.38
3,82,Paladin_748,Санкт-Петербург,3.33
4,87,Kaiser_118,Екатеринбург,3.29
5,7,Vortex_482,Екатеринбург,3.17
6,24,Striker_720,Казань,3.00
7,27,Sentinel_237,Санкт-Петербург,3.00
8,97,Sniper_285,Москва,3.00
9,94,Samurai_147,Санкт-Петербург,2.83
